In [4]:
import pandas as pd

In [7]:
df = pd.read_csv("/Users/ar.razin/HW/val_user_agg.csv")
df

,user_id,total_views,unique_views,avg_text_len_viewed,avg_word_count_viewed,total_likes,unique_likes,avg_text_len_liked,avg_word_count_liked,view_topic_business,...,view_topic_sport,view_topic_tech,like_topic_business,like_topic_covid,like_topic_entertainment,like_topic_movie,like_topic_politics,like_topic_sport,like_topic_tech,most_common_topic
0,200,307,299,1515.368078,260.628664,37,37,1500.864865,259.135135,25,...,56,9,2,6,4,14,3,7,1,covid
1,201,605,575,1350.704132,234.104132,46,46,1411.630435,245.391304,27,...,73,23,3,10,4,25,1,1,2,movie
2,202,509,481,1267.453831,220.644401,69,69,1316.840580,227.768116,16,...,65,16,1,18,1,24,15,10,0,movie
3,203,234,227,1349.623932,234.982906,43,43,1269.883721,221.069767,15,...,35,8,1,11,1,18,2,8,2,movie
4,204,53,53,1108.188679,192.490566,13,13,923.923077,156.153846,3,...,5,2,0,4,1,6,0,1,1,movie
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163200,168548,332,323,1437.186747,249.322289,18,18,1596.444444,271.388889,12,...,48,12,1,6,1,6,1,0,3,movie
163201,168549,218,218,1433.559633,249.541284,21,21,1760.095238,308.809524,12,...,22,8,0,5,0,5,7,4,0,movie
163202,168550,346,341,1386.497110,240.447977,37,37,1248.000000,219.216216,26,...,31,13,1,10,2,22,0,1,1,movie
163203,168551,447,428,1322.299776,229.982103,43,42,963.813953,168.674419,21,...,58,14,1,14,1,21,1,4,1,movie


In [6]:
from sqlalchemy import create_engine

engine = create_engine(
    "postgresql://robot-startml-ro:pheiph0hahj1Vaif@"
    "postgres.lab.karpov.courses:6432/startml"
)

205

In [10]:
cols = pd.read_csv("/Users/ar.razin/HW/val_dataset_advanced.csv", nrows = 1).columns

In [14]:
set(cols) - set(df.columns) - set(["post_id", "text"])

{'age',
 'city',
 'country',
 'exp_group',
 'gender',
 'len_diff_from_avg_view',
 'len_ratio_from_avg_view',
 'os',
 'source',
 'target',
 'text_len',
 'topic',
 'topic_match',
 'word_count'}

In [30]:
engine.connect()

In [17]:
user_data = pd.read_sql(
"""
SELECT
    *
FROM user_data
"""
, con=engine)

In [18]:
post_data = pd.read_sql(
"""
SELECT
    *
FROM post_text_df
"""
, con=engine)

In [19]:
post_features = post_data.copy()
post_features['text_len'] = post_features['text'].str.len()
post_features['word_count'] = post_features['text'].str.split().str.len()
post_features = post_features[['post_id', 'topic', 'text_len', 'word_count', 'text']]

In [20]:
post_features

,post_id,topic,text_len,word_count,text
0,1,business,1967,324,UK economy facing major risks\n\nThe UK manufa...
1,2,business,2701,448,Aids and climate top Davos agenda\n\nClimate c...
2,3,business,3408,546,Asian quake hits European shares\n\nShares in ...
3,4,business,1026,173,India power shares jump on debut\n\nShares in ...
4,5,business,889,150,Lacroix label bought by US firm\n\nLuxury good...
...,...,...,...,...,...
7018,7315,movie,803,159,"OK, I would not normally watch a Farrelly brot..."
7019,7316,movie,800,150,I give this movie 2 stars purely because of it...
7020,7317,movie,636,111,I cant believe this film was allowed to be mad...
7021,7318,movie,728,140,The version I saw of this film was the Blockbu...


In [21]:
from catboost import CatBoostClassifier

model = CatBoostClassifier().load_model("model.cbm")

In [34]:
big = df.merge(user_data, how = "inner", on = "user_id")
big

,user_id,total_views,unique_views,avg_text_len_viewed,avg_word_count_viewed,total_likes,unique_likes,avg_text_len_liked,avg_word_count_liked,view_topic_business,...,like_topic_sport,like_topic_tech,most_common_topic,gender,age,country,city,exp_group,os,source
0,200,307,299,1515.368078,260.628664,37,37,1500.864865,259.135135,25,...,7,1,covid,1,34,Russia,Degtyarsk,3,Android,ads
1,201,605,575,1350.704132,234.104132,46,46,1411.630435,245.391304,27,...,1,2,movie,0,37,Russia,Abakan,0,Android,ads
2,202,509,481,1267.453831,220.644401,69,69,1316.840580,227.768116,16,...,10,0,movie,1,17,Russia,Smolensk,4,Android,ads
3,203,234,227,1349.623932,234.982906,43,43,1269.883721,221.069767,15,...,8,2,movie,0,18,Russia,Moscow,1,iOS,ads
4,204,53,53,1108.188679,192.490566,13,13,923.923077,156.153846,3,...,1,1,movie,0,36,Russia,Anzhero-Sudzhensk,3,Android,ads
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163200,168548,332,323,1437.186747,249.322289,18,18,1596.444444,271.388889,12,...,0,3,movie,0,36,Russia,Kaliningrad,4,Android,organic
163201,168549,218,218,1433.559633,249.541284,21,21,1760.095238,308.809524,12,...,4,0,movie,0,18,Russia,Tula,2,Android,organic
163202,168550,346,341,1386.497110,240.447977,37,37,1248.000000,219.216216,26,...,1,1,movie,1,41,Russia,Yekaterinburg,4,Android,organic
163203,168551,447,428,1322.299776,229.982103,43,42,963.813953,168.674419,21,...,4,1,movie,0,38,Russia,Moscow,3,iOS,organic


In [29]:
set(model.feature_names_) - set(big.columns) - set(post_features.columns)

{'len_diff_from_avg_view', 'len_ratio_from_avg_view', 'topic_match'}

In [41]:
big.to_sql('sofja_kirjanova__features_lesson_22', con=engine, if_exists = "replace", index = False) # записываем таблицу

205

In [42]:
post_features[["post_id", "topic", "text_len", "word_count"]].to_sql(
    'sofja_kirjanova__features_lesson_22__post_features', con=engine, if_exists = "replace", index = False
)

23

In [43]:
check = pd.read_sql(
"""
SELECT
    *
FROM sofja_kirjanova__features_lesson_22
"""
, con=engine)

In [44]:
check

,user_id,total_views,unique_views,avg_text_len_viewed,avg_word_count_viewed,total_likes,unique_likes,avg_text_len_liked,avg_word_count_liked,view_topic_business,...,like_topic_sport,like_topic_tech,most_common_topic,gender,age,country,city,exp_group,os,source
0,200,307,299,1515.368078,260.628664,37,37,1500.864865,259.135135,25,...,7,1,covid,1,34,Russia,Degtyarsk,3,Android,ads
1,201,605,575,1350.704132,234.104132,46,46,1411.630435,245.391304,27,...,1,2,movie,0,37,Russia,Abakan,0,Android,ads
2,202,509,481,1267.453831,220.644401,69,69,1316.840580,227.768116,16,...,10,0,movie,1,17,Russia,Smolensk,4,Android,ads
3,203,234,227,1349.623932,234.982906,43,43,1269.883721,221.069767,15,...,8,2,movie,0,18,Russia,Moscow,1,iOS,ads
4,204,53,53,1108.188679,192.490566,13,13,923.923077,156.153846,3,...,1,1,movie,0,36,Russia,Anzhero-Sudzhensk,3,Android,ads
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163200,168548,332,323,1437.186747,249.322289,18,18,1596.444444,271.388889,12,...,0,3,movie,0,36,Russia,Kaliningrad,4,Android,organic
163201,168549,218,218,1433.559633,249.541284,21,21,1760.095238,308.809524,12,...,4,0,movie,0,18,Russia,Tula,2,Android,organic
163202,168550,346,341,1386.497110,240.447977,37,37,1248.000000,219.216216,26,...,1,1,movie,1,41,Russia,Yekaterinburg,4,Android,organic
163203,168551,447,428,1322.299776,229.982103,43,42,963.813953,168.674419,21,...,4,1,movie,0,38,Russia,Moscow,3,iOS,organic
